In [8]:
!pip install pymysql

In [9]:
from sqlalchemy import create_engine, text
import pandas as pd

USER = 'root'
PASSWORD = 'thaichauanh%40308'
HOST = 'localhost'
PORT = 3306
DB = 'olist'

engine = create_engine(f'mysql+pymysql://{USER}:{PASSWORD}@{HOST}:{PORT}/{DB}')

with engine.connect() as conn:
    result = conn.execute(text('SELECT VERSION()')).fetchone()
    print('Connected to MySQL:', result[0])

Connected to MySQL: 8.0.46


In [11]:
from sqlalchemy.types import VARCHAR, TEXT
import time

data_path = './data/raw/'
order_date_cols = [
    'order_purchase_timestamp', 'order_approved_at',
    'order_delivered_carrier_date', 'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

table_specs = {
    'customers':            ('olist_customers_dataset.csv', None),
    'geolocation':          ('olist_geolocation_dataset.csv', None),
    'order_items':          ('olist_order_items_dataset.csv', ['shipping_limit_date']),
    'order_payments':       ('olist_order_payments_dataset.csv', None),
    'order_reviews':        ('olist_order_reviews_dataset.csv',
                             ['review_creation_date', 'review_answer_timestamp']),
    'orders':               ('olist_orders_dataset.csv', order_date_cols),
    'products':             ('olist_products_dataset.csv', None),
    'sellers':              ('olist_sellers_dataset.csv', None),
    'category_translation': ('product_category_name_translation.csv', None),
}

def build_dtype_map(df):
    dtype_map = {}
    for col in df.select_dtypes('object').columns:
        # Calculate max length seen in this column (NaN-safe)
        max_len = df[col].astype(str).str.len().max()
        if pd.isna(max_len) or max_len <= 50:
            dtype_map[col] = VARCHAR(50)
        elif max_len <= 255:
            dtype_map[col] = VARCHAR(255)
        else:
            dtype_map[col] = TEXT
    return dtype_map

for table_name, (file_name, date_cols) in table_specs.items():
    start = time.time()
    df = pd.read_csv(data_path + file_name, parse_dates=date_cols)
    dtype_map = build_dtype_map(df)
    df.to_sql(table_name, engine, if_exists='replace', index=False,
              chunksize=5000, dtype=dtype_map)
    elapsed = time.time() - start
    print(f"Loaded {table_name:25s} ({len(df):>8,} rows) in {elapsed:.1f}s")

print("\nDone.")

Loaded customers                 (  99,441 rows) in 1.3s
Loaded geolocation               (1,000,163 rows) in 11.5s
Loaded order_items               ( 112,650 rows) in 2.1s
Loaded order_payments            ( 103,886 rows) in 1.3s
Loaded order_reviews             (  99,224 rows) in 2.0s
Loaded orders                    (  99,441 rows) in 2.7s
Loaded products                  (  32,951 rows) in 0.7s
Loaded sellers                   (   3,095 rows) in 0.1s
Loaded category_translation      (      71 rows) in 0.0s

Done.
